# Exoplot ENS: Complete Backend Guide & Tutorial
Welcome to the official documentation notebook for the **Exoplot ENS** scientific engine. 

This notebook is designed for discovering users. It serves as a comprehensive guide to navigating the object-oriented backend, demonstrating how to seamlessly chain database filters, process raw lightcurves, run highly optimized parallel MCMC fits, and generate publication-quality LaTeX reports.

---
## 0. Environment Setup & Imports
This cell initializes the environment, suppresses non-critical warnings (like those from Lightkurve dependencies) for a clean tutorial experience, and imports the entire Exoplot ENS backend suite.

In [1]:
# ==========================================
# 0. Environment Setup & Imports
# ==========================================
import warnings
import pandas as pd
from IPython.display import display, HTML, FileLink
from tqdm.notebook import tqdm

# Suppress non-critical warnings for a clean tutorial experience
warnings.filterwarnings('ignore') 

# Exhaustive imports from the Exoplot backend
from modules import constants
from modules.models import MassRadiusModels
from modules.catalog import ExoplanetCatalog
from modules.plotting import PlotStyle, CatalogPlotter, TransitPlotter
from modules.lightcurve import LightCurveAnalyzer
from modules.mcmc import TransitFitter
from modules.reports import ReportGenerator

print("Exoplot Backend successfully initialized.")

Exoplot Backend successfully initialized.


---
## 1. Constants & Configurations (`constants.py`)
This cell tests the global configuration file. It verifies that directory paths are correctly dynamically resolved (preventing "file not found" errors) and exposes the default astronomical mappings and MCMC baseline boundaries.

In [2]:
# ==========================================
# 1. Constants & Configurations (constants.py)
# ==========================================
print("--- DIRECTORY PATHS ---")
print(f"Base Directory: {constants.BASE_DIR}")
print(f"Data Directory: {constants.DATA_DIR}")
print(f"Models Directory: {constants.MODELS_DIR}")
print(f"Resolved NEA Path: {constants.DATA_PATHS.get('NEA')}")

print("\n--- ASTRONOMICAL CONSTANTS ---")
print(f"Spectral Types mapped: {list(constants.SPECTRAL_TYPE_TEMPERATURES.keys())}")
print(f"Habitable Zone Models: {constants.HZ_NAMES}")

print("\n--- DEFAULT MCMC CONFIGURATION ---")
print(f"Default Labels: {constants.MCMC_LABELS}")
print(f"Default Bounds: {constants.MCMC_BOUNDS}")
print(f"Batman Limb Darkening: {constants.LIMB_DARKENING_MODEL} {constants.LIMB_DARKENING_COEFFS}")

--- DIRECTORY PATHS ---
Base Directory: /Users/simon.wtmn/Desktop/Exoplot_ENS
Data Directory: /Users/simon.wtmn/Desktop/Exoplot_ENS/data
Models Directory: /Users/simon.wtmn/Desktop/Exoplot_ENS/data/theoretical_models
Resolved NEA Path: /Users/simon.wtmn/Desktop/Exoplot_ENS/data/NEA_03042026.csv

--- ASTRONOMICAL CONSTANTS ---
Spectral Types mapped: ['O', 'B', 'A', 'F', 'G', 'K', 'M', 'L', 'T']
Habitable Zone Models: ['Recent Venus', 'Runaway Greenhouse', 'Maximum Greenhouse', 'Early Mars', '5ME Runaway Greenhouse', '0.1ME Runaway Greenhouse']

--- DEFAULT MCMC CONFIGURATION ---
Default Labels: ['$R_p / R_s$', 'Inclination (deg)', '$a/R_s$', '$t_0$']
Default Bounds: [(0.001, 0.2), (89.6, 95), (4, 4.2), (-0.04, 0.04)]
Batman Limb Darkening: quadratic [0.1, 0.3]


---
### 2. Theoretical Mass-Radius Models (`models.py`)
This cell tests the `MassRadiusModels` class. It demonstrates how to inspect the catalog of available theoretical curves (like Zeng, Aguichine, Lopez & Fortney) and how the code seamlessly loads diverse text/CSV formats into standardized Pandas DataFrames.

In [3]:
# ==========================================
# 2. Theoretical Mass-Radius Models (models.py)
# ==========================================
mr_models = MassRadiusModels()

# Attribute exposure
print(f"Model Directory set to: {mr_models.models_directory}")

# Method: list_available_models()
all_models = mr_models.list_available_models()
print(f"\nTotal models loaded in catalog: {len(all_models)}")

# Method: get_model_label()
sample_key = 'zeng_earth'
print(f"Legend label for '{sample_key}': {mr_models.get_model_label(sample_key)}")

# Method: get_model_curve()
earth_curve = mr_models.get_model_curve(sample_key)
print("\nExtracted DataFrame for Earth-like model:")
display(earth_curve)

Model Directory set to: /Users/simon.wtmn/Desktop/Exoplot_ENS/data/theoretical_models

Total models loaded in catalog: 50
Legend label for 'zeng_earth': Zeng+2019: Earth-like

Extracted DataFrame for Earth-like model:


,mass,radius
0,0.0030,0.1648
1,0.0042,0.1831
2,0.0059,0.2036
3,0.0082,0.2267
4,0.0114,0.2524
5,0.0159,0.2810
6,0.0221,0.3128
7,0.0306,0.3476
8,0.0420,0.3854
9,0.0575,0.4265


---
### 3. Exoplanet Catalog Manager (`catalog.py`)
This cell tests the `ExoplanetCatalog` class, which handles the NASA Exoplanet Archive data. It demonstrates the power of **Method Chaining**, allowing us to apply an exhaustive list of physical and orbital filters in a single block, and tests the memory-safe `.reset()` method.

In [4]:
# ==========================================
# 3. Exoplanet Catalog Manager (catalog.py)
# ==========================================
# Instantiation
catalog = ExoplanetCatalog(dataset_name='NEA')

# Attribute exposure
print(f"Catalog Name: {catalog.name}")
print(f"Original Data Size: {catalog.original_df.shape}")
print(f"Current Working Data Size: {catalog.df.shape}")

# Exhaustive Method Chaining (Testing EVERY filter)
print("\nApplying exhaustive filter chain...")
catalog.filter_discovery(mission='Kepler', year_min=2010, year_max=2024, kp_max=16) \
       .filter_stellar(st_type='M', teff_min=2500, teff_max=4000, rad_max=0.6) \
       .filter_spectral_type('M') \
       .filter_planet(mass_max=20, rade_max=3.0, eqt_max=1000) \
       .filter_orbit(period_max=50, eccentricity_max=0.5) \
       .filter_system(multiplicity_min=1) \
       .filter_fulton_gap()

# Method: get_data()
filtered_df = catalog.get_data()
print(f"Filtered Data Size: {filtered_df.shape}")
display(filtered_df[['pl_name', 'hostname', 'pl_bmasse', 'pl_rade']].head(3))

# Method: reset()
catalog.reset()
print(f"Size after reset(): {catalog.df.shape} (Matches original: {len(catalog.df) == len(catalog.original_df)})")

Catalog Name: NEA
Original Data Size: (6128, 219)
Current Working Data Size: (6128, 219)

Applying exhaustive filter chain...
Filtered Data Size: (30, 219)


,pl_name,hostname,pl_bmasse,pl_rade
0,Kepler-125 b,Kepler-125,6.21,2.37
1,Kepler-125 c,Kepler-125,0.33,0.74
2,Kepler-138 b,Kepler-138,0.07,0.64


Size after reset(): (6128, 219) (Matches original: True)


---
### 4. Catalog Visualization (`plotting.py`)
This cell tests the `CatalogPlotter` and the `PlotStyle` utility. It generates interactive Plotly HTML figures for macroscopic populations, testing color gradients, log scales, model overlays, 2D Gaussian density smoothing, and dynamic Light/Dark theme switching.

In [5]:
# ==========================================
# 4. Catalog Visualization (plotting.py)
# ==========================================
cat_plotter = CatalogPlotter()

# Testing PlotStyle utility directly
print(f"PlotStyle label translation for 'pl_orbeccen': {PlotStyle.get_label('pl_orbeccen')}")

# Method: plot_scatter() (Testing highlight, color_by, log scales, models, and light theme)
print("\nGenerating Scatter Plot (Light Theme)...")
scatter_html = cat_plotter.plot_scatter(
    df=filtered_df, x_col='pl_bmasse', y_col='pl_rade', 
    color_by='st_teff', highlight_planets=['Kepler-138 c'], 
    log_x=True, log_y=False, theme='dark', 
    overlay_models=['zeng_earth', 'Water World']
)
display(HTML(scatter_html))

# Method: plot_density() (Testing gaussian smoothing, viridis map, and dark theme)
print("Generating Density Plot (Dark Theme)...")
density_html = cat_plotter.plot_density(
    df=catalog.original_df, x_col='pl_orbper', y_col='pl_rade', 
    log_x=True, log_y=False, cmap='Viridis', theme='dark'
)
display(HTML(density_html))

# Method: plot_histogram() (Testing 1D binning)
print("Generating Histogram...")
hist_html = cat_plotter.plot_histogram(
    df=filtered_df, column='pl_rade', bins=30, log_x=False, theme='dark'
)
display(HTML(hist_html))

PlotStyle label translation for 'pl_orbeccen': Eccentricity

Generating Scatter Plot (Light Theme)...


Generating Density Plot (Dark Theme)...


Generating Histogram...


---
### 5. Lightcurve Processing (`lightcurve.py`)
Moving to individual systems, this cell tests the `LightCurveAnalyzer`. It verifies the automated MAST archive query, data downloading and cleaning, Box Least Squares (BLS) periodogram computation, and phase-folding mechanics.

In [6]:
# ==========================================
# 5. Lightcurve Processing (lightcurve.py)
# ==========================================
TARGET = "WASP 76"
analyzer = LightCurveAnalyzer(TARGET)

# Method: search()
search_df = analyzer.search()
print(f"Search Results for {TARGET}:")
display(search_df)

# Method: download_and_clean()
analyzer.download_and_clean(index=0)
print(f"\nRaw Lightcurve Type: {type(analyzer.raw_lc)}")
print(f"Cleaned Lightcurve Type: {type(analyzer.clean_lc)}")

# Method: compute_periodogram()
analyzer.compute_periodogram()
print("\nExtracted BLS Attributes:")
print(f" - best_period: {analyzer.best_period:.6f} days")
print(f" - best_freq: {analyzer.best_freq:.6f} 1/days")
print(f" - best_power: {analyzer.best_power:.2f}")
print(f" - epoch_time: {analyzer.epoch_time:.6f} BJD")

# Method: fold_lightcurve()
analyzer.fold_lightcurve(harmonic=1)
print(f"Folded Lightcurve generated: {analyzer.folded_lc is not None}")

# Method: get_mcmc_data()
time, flux, err, period, t0 = analyzer.get_mcmc_data(folded=True)
print(f"MCMC Data Extracted - Time array shape: {time.shape}")

Search Results for WASP 76:


,mission,year,author,exptime,target_name,distance
0,TESS Sector 30,2020,SPOC,120.0,293435336,0.0
1,TESS Sector 42,2021,SPOC,120.0,293435336,0.0
2,TESS Sector 43,2021,SPOC,120.0,293435336,0.0
3,TESS Sector 97,2025,SPOC,20.0,293435336,0.0
4,TESS Sector 97,2025,SPOC,120.0,293435336,0.0
5,TESS Sector 30,2020,TESS-SPOC,600.0,293435336,0.0
6,TESS Sector 42,2021,TESS-SPOC,600.0,293435336,0.0
7,TESS Sector 43,2021,TESS-SPOC,600.0,293435336,0.0
8,TESS Sector 30,2020,QLP,600.0,293435336,0.0
9,TESS Sector 42,2021,QLP,600.0,293435336,0.0



Raw Lightcurve Type: <class 'lightkurve.lightcurve.TessLightCurve'>
Cleaned Lightcurve Type: <class 'lightkurve.lightcurve.TessLightCurve'>

Extracted BLS Attributes:
 - best_period: 1.812384 days
 - best_freq: 0.551759 1/days
 - best_power: 206057.81
 - epoch_time: 2137.599663 BJD
Folded Lightcurve generated: True
MCMC Data Extracted - Time array shape: (16085,)


---
### 6. MCMC Bayesian Inference (`mcmc.py`)
This cell tests the powerhouse of the backend: `TransitFitter`. It demonstrates the dynamic nature of the code by fitting **5 parameters** (including eccentricity) instead of the default 4. It tests the L-BFGS-B pre-optimization and the `emcee` parallel processing utilizing all CPU cores.

In [7]:
# ==========================================
# 6. RAW MCMC Bayesian Inference (WASP-76 b)
# ==========================================
# Notice folded=False! We are fitting the raw time series!
time, flux, err, period_guess, t0_guess = analyzer.get_mcmc_data(folded=False)

# On fit 6 paramètres (On exclut t0)
custom_params = ['rp', 'inc', 'a', 'per', 'u1', 'u2']

custom_labels = [
    r"$R_p / R_s$", 
    r"Inclination (deg)", 
    r"$a/R_s$", 
    r"Period (days)", 
    r"$u_1$", 
    r"$u_2$"
]

custom_bounds = [
    (0.08, 0.15),                                # rp/rs
    (80.0, 90.0),                                # inclination
    (3.0, 6.0),                                  # a/rs
    (period_guess - 0.05, period_guess + 0.05),  # period
    (0.0, 1.0),                                  # u1
    (0.0, 1.0)                                   # u2
]

custom_x0 = [0.106, 89.0, 4.1, period_guess, 0.3, 0.2]

fitter = TransitFitter(
    time=time, flux=flux, flux_err=err, period=period_guess, t0=t0_guess,
    fitted_params=custom_params, custom_bounds=custom_bounds, 
    custom_x0=custom_x0, custom_labels=custom_labels
)

best_guess = fitter.optimize_initial_guess()
print("L-BFGS-B Initial Optimization Result:", best_guess)

N_WALKERS, N_STEPS = 32, 1500
pbar = tqdm(total=N_STEPS, desc="Raw Data MCMC")
last_step = [0]
def update_pbar(current, total):
    pbar.update(current - last_step[0])
    last_step[0] = current

results = fitter.run_mcmc(nwalkers=N_WALKERS, nsteps=N_STEPS, progress_callback=update_pbar, use_multiprocessing=True)
pbar.close()

print("\nFit Complete. Derived Parameters:")
for param, (median, upper, lower) in results.items():
    print(f" {param}: {median:.5f} (+{upper:.5f} / -{lower:.5f})")

L-BFGS-B Initial Optimization Result: [ 0.10599856 88.99999363  4.10010508  1.81238664  0.29998341  0.19998894]


Raw Data MCMC:   0%|          | 0/1500 [00:00<?, ?it/s]


Fit Complete. Derived Parameters:
 $R_p / R_s$: 0.10606 (+0.00071 / -0.00062)
 Inclination (deg): 88.99998 (+0.00068 / -0.00086)
 $a/R_s$: 4.09997 (+0.00066 / -0.00055)
 Period (days): 1.81177 (+0.00060 / -0.00083)
 $u_1$: 0.29995 (+0.00056 / -0.00029)
 $u_2$: 0.20008 (+0.00097 / -0.00086)


---
### 7. Transit Plotting (`plotting.py`)
This cell tests the `TransitPlotter` methods. It creates interactive Plotly interfaces for the BLS periodogram, the folded lightcurve overlaid with the high-resolution `batman` theoretical model, and the interactive MCMC diagnostic tools (Traces and Corner plots).

In [8]:
# ==========================================
# 7. Transit Plotting (plotting.py)
# ==========================================
# Method: plot_periodogram()
print("Generating Periodogram...")
per_html = TransitPlotter.plot_periodogram(
    x=analyzer.periodogram.period.value, y=analyzer.periodogram.power.value,
    title=f"BLS Power Spectrum", xaxis_type='period', theme='dark'
)
display(HTML(per_html))

# Method: get_best_model_curve() & plot_lightcurve()
print("Generating Folded Fit Overlay...")
model_time, model_flux = fitter.get_best_model_curve(num_points=3000)
fit_html = TransitPlotter.plot_lightcurve(
    x=time, y=flux, err=err, model_x=model_time, model_y=model_flux,
    title="MCMC Best Fit Overlay", style='scatter', bins=50, theme='dark'
)
display(HTML(fit_html))

# Method: plot_mcmc_traces()
print("Generating MCMC Traces...")
traces_html = TransitPlotter.plot_mcmc_traces(fitter.flat_samples, fitter.labels, theme='dark')
display(HTML(traces_html))

# Method: plot_mcmc_corner()
print("Generating Plotly Corner Plot...")
corner_html = TransitPlotter.plot_mcmc_corner(fitter.flat_samples, fitter.labels, theme='dark')
display(HTML(corner_html))

Generating Periodogram...


Generating Folded Fit Overlay...


Generating MCMC Traces...


Generating Plotly Corner Plot...


---
### 8. Publication-Quality PDF Reports (`reports.py`)
The final module tested is the `ReportGenerator`. This confirms the ability to leverage Matplotlib, GridSpec, and LaTeX to output static, publication-ready PDF sheets directly to the `results/` directory, formatted to TESS Data Validation standards.

In [ ]:
# ==========================================
# 8. PDF Report Generation (reports.py)
# ==========================================
# Instantiation (Exposing attributes)
report_gen = ReportGenerator(website_name="Exoplot", use_latex=True)
print(f"PDFs will be saved to: {report_gen.results_dir}")
print(f"Using LaTeX Rendering: {report_gen.use_latex}")

# Method: generate_mcmc_summary_report()
summary_pdf = f"{analyzer.target_name}_Summary.pdf"
report_gen.generate_mcmc_summary_report(summary_pdf, analyzer, fitter)

# Method: generate_mcmc_diagnostic_report()
diag_pdf = f"{analyzer.target_name}_Diagnostics.pdf"
report_gen.generate_mcmc_diagnostic_report(diag_pdf, analyzer, fitter)

# Method: generate_catalog_report()
cat_pdf = "Catalog_Report.pdf"
plot_templates = [
    {'x': 'pl_bmasse', 'y': 'pl_rade', 'log_x': True, 'type': 'scatter'},
    {'x': 'pl_orbper', 'y': 'pl_orbeccen', 'log_x': True, 'type': 'scatter'},
    {'x': 'pl_rade', 'type': 'histogram'},
    {'x': 'st_teff', 'type': 'histogram'}
]
report_gen.generate_catalog_report(cat_pdf, filtered_df, plot_templates, title="Exhaustive Catalog Test")

print("\All Reports Generated Successfully!")

PDFs will be saved to: /Users/simon.wtmn/Desktop/Exoplot_ENS/results
Using LaTeX Rendering: True
\All Reports Generated Successfully!
